# Análise exploratória — Emissões de GEE do Rio de Janeiro (2012–2024)
Projeto final · Linguagem de Programação · Fonte: Data.Rio/IPP, Tabela 3696.

Execute da raiz do repositório (`jupyter lab`) após `python -m src.etl`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src import analysis

df = analysis.carregar(ROOT / "data" / "processed" / "emissoes_long.csv.gz")
df.head()

## 1. Estrutura e qualidade dos dados

In [ ]:
print(df.shape)
print(df.dtypes)
print("Ausentes por nível:")
print(df.groupby("nivel")["valor_tco2e"].apply(lambda s: s.isna().sum()))
print(df[df.valor_tco2e < 0].categoria.unique())

### Validação: os 5 setores somam o TOTAL publicado?

In [ ]:
s = analysis.tabela_setores(df)
tot = df[(df.dimensao=="escopo") & (df.nivel=="total") & (df.componente=="Total")].set_index("ano")["valor_tco2e"]
print("Diferença máxima (t CO2e):", (s.sum(axis=1) - tot).abs().max())

## 2. RQ1 — Total e composição por setor

In [ ]:
tot = s.sum(axis=1)
resumo = pd.DataFrame({"total": tot, "var_%_ano": tot.pct_change()*100})
print(resumo.round(1))
part = s.div(tot, axis=0) * 100
part.loc[[2012, 2019, 2024]].T.round(1)

In [ ]:
ax = (s/1e6).plot.area(figsize=(9,4.5), color=[analysis.COR_SETOR[c] for c in s.columns])
ax.set_ylabel("Milhões de t CO2e"); ax.set_title("Emissões por setor"); plt.show()

Coeficiente de variação (CV) por setor = desvio-padrão amostral (`ddof=1`, padrão do pandas) / média, × 100, sobre os 13 valores anuais de 2012–2024. Inclui a tendência da série e é instável quando a média é pequena:

In [ ]:
(s.std()/s.mean()*100).round(1).sort_values(ascending=False)

## 3. RQ2 — Escopos e o papel do Escopo 2

In [ ]:
esc = analysis.tabela_total(df, "escopo")
print((esc.std()/esc.mean()*100).round(1))
sin = analysis.serie_sin(df)
print("corr(Escopo 2, geração p/ SIN) =", round(esc["Escopo 2"].corr(sin), 2))
(esc[["Escopo 1","Escopo 2","Escopo 3"]]/1e6).plot(marker="o", figsize=(9,4)); plt.ylabel("Mt CO2e"); plt.show()

## 4. RQ3 — Subsetores e gases

In [ ]:
sub = analysis.tabela_subsetores(df)
delta = (sub[2024] - sub[2012]).sort_values()
print((delta.head(5)/1e6).round(2)); print((delta.tail(5)/1e6).round(2))
print("Soma das 3 maiores quedas (Mt):", round(delta.head(3).sum()/1e6, 2), "| variação líquida (Mt):", round(delta.sum()/1e6, 2))

In [ ]:
gas = analysis.tabela_total(df, "gas")
(gas[["CO2","CH4","N2O"]].div(gas["Total"], axis=0)*100).loc[[2012, 2018, 2024]].round(2)

## 5. Reprodução das figuras do artigo
`python -m src.analysis` regenera os PNGs em `docs/assets/`.